# REDACT Library — Full Pipeline Demo

End-to-end dataset generation using the high-level pipeline functions.

This notebook demonstrates the whole flow in one place. The same stages are also
available as focused, standalone notebooks:
`constitution_generation.ipynb`, `content_moderation.ipynb`, `jailbreak_augmentation.ipynb`.

Stages:
0. **Constitution** — generate structured category hierarchies (harmful, benign, dual-use)
0b. **Constitution-to-input** — expand constitution entries into full realistic prompts
0d. **Constitution jailbreaks** — augment constitution prompts with technique combinations
1. **Content moderation inputs** — generate harmful prompts across taxonomy categories
2. **Output responses** — generate model responses for the input samples
3. **Jailbreaks** — apply obfuscation, hacking, and manipulation techniques
4. **Complete dataset** — merge everything into a single CSV

Requires `VENICE_API_KEY` and `ANTHROPIC_API_KEY` set in environment or `.env` file.

In [ ]:
from redact import set_seed
set_seed(42)

## Configuration

Adjust these settings before running. Small values are set for demo purposes.

In [ ]:
# --- Pipeline settings ---
MODEL = "venice-uncensored-vllm"          # Generation model
BASE_URL = "https://api.venice.ai/api/v1"
CONSTITUTION_MODEL = "claude-opus-4-6"  # Constitution generation model

# Constitution
CONSTITUTION_CATEGORIES = 5          # Constitution categories per type per taxonomy category
CONSTITUTION_ENTRY_TYPES = None       # None = all 4 types (harmful, benign, dual_use_benign, dual_use_harmful)
CONSTITUTION_STANDALONE_BENIGN = True # Generate category-free benign entries (single LLM call)
CONSTITUTION_STANDALONE_BENIGN_CATEGORIES = 10  # Number of benign categories in standalone call

# Constitution-to-input
CONSTITUTION_INPUT_STYLE = ["long", "short"]    # Template style: "long" (detailed, 2-5 sentences) or "short" (concise, 5-20 words)
CONSTITUTION_INPUT_SAMPLES_PER_ENTRY = 2  # Number of prompts to generate per constitution entry
CONSTITUTION_INPUT_ENTRY_TYPES = None   # Which entry types to expand (None = all)

# Content moderation input
SAMPLES_PER_CATEGORY = 15            # Target accepted samples per category
NUM_CATEGORIES = 3                   # Number of categories to process (None = all)
USE_METAPROMPT = True                # True = LLM generates descriptions + seeds
NUM_SEEDS = 8                        # Number of seed prompts to generate (metaprompt mode)
SAMPLES_PER_REQUEST = 5              # Samples requested per LLM call
FRESH_RUN = True                     # Clear existing data before generating (avoids inflated rejections)

# Output responses
MAX_OUTPUT_SAMPLES = 10              # Limit output generation (None = all)

# Jailbreaks
JAILBREAK_MAX_COMPLEXITY = 6         # Max total complexity per combination
JAILBREAK_MAX_OBFUSCATIONS = 2       # Max obfuscation families per combination
JAILBREAK_PURE_ONLY = False          # True = no LLM-dependent techniques (no API calls)

## Step 0: Generate Constitution

Generates a structured category hierarchy for constitutional classifier training.
For each taxonomy category, creates entries across 4 severity levels:
- **Harmful** — absolutely harmful, always flag
- **Dual-use harmful** — borderline harmful framing, debatable
- **Dual-use benign** — borderline benign framing, could look harmful
- **Benign** — absolutely benign, never flag (hard negatives)

Optionally generates **standalone benign** entries in a single category-free LLM call
(no taxonomy influence). These are saved separately to `general_benign.csv`.

Each entry can later seed N input samples. Saved to `Data_cache/constitution/`.

Requires `ANTHROPIC_API_KEY` (uses Claude Opus for generation).

In [ ]:
from redact import generate_constitution

constitution = generate_constitution(
    taxonomy="content_moderation_categories",
    entry_types=CONSTITUTION_ENTRY_TYPES,
    num_categories=CONSTITUTION_CATEGORIES,
    model=CONSTITUTION_MODEL,
    num_taxonomy_categories=None,  # All taxonomy categories
    include_standalone_benign=CONSTITUTION_STANDALONE_BENIGN,
    standalone_benign_categories=CONSTITUTION_STANDALONE_BENIGN_CATEGORIES,
)

print(f"\nGenerated {len(constitution)} constitution entries")
if not constitution.empty:
    print("\n=== By Entry Type ===")
    print(constitution["entry_type"].value_counts().to_string())
    print("\n=== By Source Category ===")
    print(constitution["source_category"].value_counts().to_string())
constitution.head(10)

## Step 0b: Constitution-to-Input Generation

Expands constitution entry descriptions into full realistic input prompts using
the composable `generate_inputs(constitution_df=...)` API (constitution-seeded mode).

Each constitution entry has a short `sample_description` (e.g., "Instructions for
making pipe bombs"). This step uses an LLM to expand each description into
`CONSTITUTION_INPUT_SAMPLES_PER_ENTRY` full prompts.

Two template styles are available:
- **long** — detailed, multi-sentence prompts (2-5 sentences with specific scenarios)
- **short** — concise, direct prompts (5-20 words)

New styles can be added by creating a template in `prompts/input/generation/from_constitution/{style}/`.

Output is saved in standard content moderation format to `Datasets/constitution_inputs/`
with constitution metadata preserved (category, subcategory, entry type, etc.).

> Note: `generate_inputs_from_constitution()` is deprecated — prefer
> `generate_inputs(constitution_df=...)` as below.

In [ ]:
import pandas as pd
from redact import generate_inputs, get_output_dir

# An explicit dataset_dir keeps constitution inputs isolated under
# Datasets/constitution_inputs/{category}/ so the merge step below can scan them.
CONSTITUTION_INPUTS_DIR = get_output_dir() / "Datasets" / "constitution_inputs"

# If CONSTITUTION_INPUT_STYLE is a list, runs once per style and concatenates.
styles = CONSTITUTION_INPUT_STYLE if isinstance(CONSTITUTION_INPUT_STYLE, list) else [CONSTITUTION_INPUT_STYLE]
all_constitution_inputs = []

for style in styles:
    result = generate_inputs(
        constitution_df=constitution,        # triggers constitution-seeded mode
        style=style,
        samples_per_entry=CONSTITUTION_INPUT_SAMPLES_PER_ENTRY,
        entry_types=CONSTITUTION_INPUT_ENTRY_TYPES,
        model=MODEL,
        dataset_dir=CONSTITUTION_INPUTS_DIR,
        batch_size=32,
    )
    print(f"\n[{style}] Generated {len(result)} accepted prompts")
    all_constitution_inputs.append(result)

constitution_inputs = pd.concat(all_constitution_inputs, ignore_index=True) if all_constitution_inputs else pd.DataFrame()

print(f"\nTotal: {len(constitution_inputs)} accepted prompts from constitution entries")
if not constitution_inputs.empty:
    print("\n=== By Category ===")
    print(constitution_inputs["category"].value_counts().to_string())
    if "entry_type" in constitution_inputs.columns:
        print("\n=== By Entry Type ===")
        print(constitution_inputs["entry_type"].value_counts().to_string())
constitution_inputs.head(10)

## Step 0c: Merge Constitution Inputs

Merges all per-category CSVs from `Datasets/constitution_inputs/` into a single clean CSV.
Filters out rejected samples and drops generation-only columns (turn, source, template_style,
reasoning, source_group_tag). Renames `sample` → `prompt` to match the standard dataset format.
Saves to `Datasets/constitution_inputs_merged.csv`.

In [ ]:
from redact.dataset import merge_constitution_input_csvs

merged_path = get_output_dir() / "Datasets" / "constitution_inputs_merged.csv"

constitution_inputs_merged = merge_constitution_input_csvs(
    base_dir=CONSTITUTION_INPUTS_DIR,
    output_path=merged_path,
)

print(f"Merged {len(constitution_inputs_merged)} accepted samples -> {merged_path}")
print(f"Columns: {list(constitution_inputs_merged.columns)}")
if not constitution_inputs_merged.empty:
    print("=== By Category ===")
    print(constitution_inputs_merged["category"].value_counts().to_string())
    print("=== By Entry Type ===")
    print(constitution_inputs_merged["entry_type"].value_counts().to_string())
constitution_inputs_merged.head(10)

## Step 0d: Jailbreaks on Constitution Inputs

Augments the constitution-seeded prompts with jailbreak technique combinations using the
high-level `generate_jailbreaks()` orchestrator (plan + batched execute). Each prompt is
assigned a randomly sampled valid combination; the sampler enforces the compatibility rules
from `combination_spec.json`, and LLM-dependent steps are pooled per model per round.

Set `JAILBREAK_PURE_ONLY=True` for a no-API smoke run. Output goes to a separate
`Datasets/constitution_jailbreaks.csv` so it doesn't clash with the Step 3 run.

In [ ]:
from redact import generate_jailbreaks

COMBINATION_ENTRY_TYPES = ["harmful", "dual_use_harmful"]
COMBINATION_MAX_SAMPLES = 20      # None = all

# Load constitution inputs (from memory or the merged CSV on disk)
if "constitution_inputs_merged" not in vars() or constitution_inputs_merged.empty:
    _merged_path = get_output_dir() / "Datasets" / "constitution_inputs_merged.csv"
    constitution_inputs_merged = pd.read_csv(_merged_path) if _merged_path.exists() else pd.DataFrame()

if constitution_inputs_merged.empty:
    print("No constitution_inputs_merged found — run Step 0c first.")
    constitution_jailbreaks = pd.DataFrame()
else:
    source = constitution_inputs_merged
    if COMBINATION_MAX_SAMPLES:
        source = source.head(COMBINATION_MAX_SAMPLES)
    constitution_jailbreaks = generate_jailbreaks(
        inputs=source,
        output_path=get_output_dir() / "Datasets" / "constitution_jailbreaks.csv",
        max_complexity=JAILBREAK_MAX_COMPLEXITY,
        max_obfuscations=JAILBREAK_MAX_OBFUSCATIONS,
        pure_only=JAILBREAK_PURE_ONLY,
        entry_types=COMBINATION_ENTRY_TYPES,
        model=MODEL,
    )
    print(f"\nGenerated {len(constitution_jailbreaks)} jailbreak rows")
    if not constitution_jailbreaks.empty:
        print("\n=== Top techniques used ===")
        print(constitution_jailbreaks["technique"].value_counts().head(10).to_string())
constitution_jailbreaks.head(5)

## Step 1: Generate Content Moderation Inputs

Standalone meta-prompt mode: generates input prompts across taxonomy categories without
constitution seeding. Saved per-category to `Datasets/{category}/samples.csv`.

In [ ]:
from redact import generate_inputs

inputs = generate_inputs(
    samples_per_category=SAMPLES_PER_CATEGORY,
    num_categories=NUM_CATEGORIES,
    use_metaprompt=USE_METAPROMPT,
    num_seeds=NUM_SEEDS,
    samples_per_request=SAMPLES_PER_REQUEST,
    model=MODEL,
    base_url=BASE_URL,
    fresh=FRESH_RUN,
)

print(f"\nGenerated {len(inputs)} accepted input samples")
inputs.head(10)

## Step 2: Generate Output Responses

Runs the model on each input sample to generate a response.
Saved to `Datasets/output_responses.csv`.

In [ ]:
from redact import generate_outputs
from redact.dataset import merge_all
import pandas as pd

if "inputs" not in vars() or inputs.empty:
    inputs = merge_all(accepted_only=True)
    if inputs.empty:
        print("No inputs available — run Step 1 first.")
    else:
        print(f"Loaded {len(inputs)} input samples from disk")

outputs = pd.DataFrame()
if not inputs.empty:
    outputs = generate_outputs(
        inputs=inputs,
        model=MODEL,
        base_url=BASE_URL,
        max_samples=MAX_OUTPUT_SAMPLES,
    )
    print(f"\nGenerated {len(outputs)} output responses")
outputs.head(5)

## Step 3: Generate Jailbreaks

Applies jailbreak techniques to the content-moderation input prompts. Techniques include:
- **Obfuscation**: encoding, translation (low-resource languages), structural wrapping, ascii art, tokenbreak, adversarial suffixes
- **Hacking**: persona roleplay, hypothetical framing, authority obedience, AVI, deep inception
- **Manipulation**: FSH (few-shot hacking), DAP (distract and persuade)

Saved to `Datasets/jailbreaks.csv` (with a resumable `.manifest.jsonl` beside it).

In [ ]:
from redact import generate_jailbreaks, get_output_dir
from redact.dataset import merge_all
import pandas as pd

if "inputs" not in vars() or inputs.empty:
    inputs = merge_all(accepted_only=True)
    if inputs.empty:
        print("No inputs available - run Step 1 first.")
    else:
        print(f"Loaded {len(inputs)} input samples from disk")

jailbreaks = pd.DataFrame()
if not inputs.empty:
    jailbreaks = generate_jailbreaks(
        inputs=inputs,
        max_complexity=JAILBREAK_MAX_COMPLEXITY,
        max_obfuscations=JAILBREAK_MAX_OBFUSCATIONS,
        pure_only=JAILBREAK_PURE_ONLY,
        model=MODEL,
        base_url=BASE_URL,
    )
    print(f"\nGenerated {len(jailbreaks)} jailbreak samples")

jailbreaks.head(10)

## Step 4: Build Complete Dataset

Merges all generated data (inputs, outputs, jailbreaks) from their saved CSVs into a single dataset.

In [ ]:
from redact import build_dataset

dataset = build_dataset()

print(f"\nComplete dataset: {len(dataset)} samples")
dataset.head(20)

## Dataset Statistics

In [ ]:
if "dataset" not in vars() or dataset.empty:
    from redact import build_dataset
    dataset = build_dataset()

if not dataset.empty:
    print("=== By Dataset Type ===")
    if "dataset_type" in dataset.columns:
        print(dataset["dataset_type"].value_counts().to_string())

    print("\n=== By Category ===")
    if "category" in dataset.columns:
        print(dataset["category"].value_counts().to_string())

    print("\n=== By Technique (jailbreaks only) ===")
    if "technique" in dataset.columns:
        jb = dataset[dataset["dataset_type"] == "jailbreak"] if "dataset_type" in dataset.columns else dataset
        print(jb["technique"].value_counts().to_string())
else:
    print("No data generated yet.")

---

# Custom Taxonomy

You can define your own taxonomy and run the same pipeline on it. This is useful for generating datasets for specific domains beyond content moderation.

In [ ]:
from redact import create_taxonomy

taxonomy = create_taxonomy(
    name="financial_harm",
    description="Prompts related to financial fraud and exploitation.",
    categories={
        "Investment Scams": {
            "description": "Prompts about fraudulent investment schemes, Ponzi schemes, pump-and-dump, fake ICOs.",
            "subcategories": ["Ponzi Schemes", "Pump and Dump", "Fake Crypto"],
        },
        "Identity Theft": {
            "description": "Prompts about stealing personal information for financial gain, phishing, social engineering.",
            "subcategories": ["Phishing", "Social Engineering", "Document Forgery"],
        },
    },
)

print(f"\nTaxonomy created with {len(taxonomy['categories'])} categories")

In [ ]:
custom_inputs = generate_inputs(
    taxonomy=taxonomy,
    samples_per_category=5,
    num_categories=2,
    model=MODEL,
    base_url=BASE_URL,
    fresh=True,
)

print(f"\nGenerated {len(custom_inputs)} samples for custom taxonomy")
custom_inputs.head()